In [9]:
## Reformat the files and save them 
import pandas as pd
from pathlib import Path

class TrajectoryFormatter:
    def __init__(self, input_dir, output_dir):
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
    
        self.columns = columns = [
            "traj_id", "traj_num", "year", "month", "day", "hour",
            "step", "timestep", "hour_back",
            "lat", "lon", "alt",
            "pressure", "theta", "air_temp",
            "rainfall", "mixdepth", "relhum", "spchumid"
        ]
    
    def _read_file(self,filepath):
        #read a file, split the lines and prepare for reformatting
        traj_lines = []
        with open(filepath) as f:
            lines = f.readlines()
            for line in lines:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                try:
                    nums = [float(p) for p in parts]
                    if len(nums) >= 19:
                        traj_lines.append(nums[:19])
                except ValueError:
                    continue
            return traj_lines

    def format_file(self, filepath):
            #process one file and save as csv
        traj_lines = self._read_file(filepath)
        if traj_lines:
            df = pd.DataFrame(traj_lines, columns=self.columns)
            out_file = self.output_dir / f"{filepath.stem}.csv"
            df.to_csv(out_file, index=False)
            print(f"Saved {out_file}")
        else:
            print(f"No numeric data found in {filepath.name}")
    
    def format_all(self,pattern="colgateaug*"):
        #process all files in dir
        for filepath in self.input_dir.glob(pattern):
            if filepath.name.endswith(".Zone.Identifier"):
                continue
            self.format_file(filepath)

if __name__ == "__main__":
    input_dir = "/home/bvissel/trajectory-analysis/trajectories/colgate/"
    output_dir = "/home/bvissel/trajectory-analysis/trajectories/formatted_data/"

    formatter = TrajectoryFormatter(input_dir, output_dir)
    formatter.format_all()




            


Saved /home/bvissel/trajectory-analysis/trajectories/formatted_data/colgateaug1000summer2012082117.csv
Saved /home/bvissel/trajectory-analysis/trajectories/formatted_data/colgateaug0500summer2012080723.csv
Saved /home/bvissel/trajectory-analysis/trajectories/formatted_data/colgateaug1000summer2012080917.csv
Saved /home/bvissel/trajectory-analysis/trajectories/formatted_data/colgateaug0500summer2012080911.csv
No numeric data found in colgateaug1000summer2012081111:Zone.Identifier
Saved /home/bvissel/trajectory-analysis/trajectories/formatted_data/colgateaug1500summer2012082117.csv
Saved /home/bvissel/trajectory-analysis/trajectories/formatted_data/colgateaug0500summer2012082323.csv
No numeric data found in colgateaug0500summer2012082311:Zone.Identifier
Saved /home/bvissel/trajectory-analysis/trajectories/formatted_data/colgateaug1500summer2012081923.csv
Saved /home/bvissel/trajectory-analysis/trajectories/formatted_data/colgateaug1500summer2012082511.csv
No numeric data found in colgate

In [ ]:
#now the files are formatted in a class system
# now can import those files, read data, and define functions for calculations

import pandas as pd
import numpy as np
from math import *

class TrajectoryCalculations:
    def __init__(self, timestamp, datafile):
        self.timestamp = timestamp
        self.datafile = pd.read_csv(datafile)
    
    def distance_travelled(self):
        #access the 10th column for lat; [1] for first, [-1] for last
        dist_travel = 0
        for i in range(len(self.datafile['lat'])):
            try:
                lat1 = self.datafile['lat'].iloc[i]
                lat2 = self.datafile['lat'].iloc[i+1]
                lon1 = self.datafile['lon'].iloc[i]
                lon2 = self.datafile['lon'].iloc[i+1]
                print(f'set{i}:{lat1},{lon1}')

                #convert decimal degrees to radians
                lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
                
                #haversine formula
                dlon = lon2 - lon1 
                dlat = lat2 - lat1 
                a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
                c = 2 * asin(sqrt(a)) 
                r = 6371 # Radius of earth in kilometers. Use 3956 for miles. Determines return value units.
                base = c * r

                print(f'set{i}: base horizontal distance: {base}')
                dist_travel = dist_travel + base
                print(f' new total distance: {dist_travel}')

            except IndexError:
                print('all calculated')
                break
        
        return dist_travel


                
            #formula
            #dist_onetimepoint = answer formula
            #print(dist_onetimepoint)
            #append.dist_travel(dist_onetimepoint)
            #print(dist_travel)

firstTraj = TrajectoryCalculations('Aug20120801', '/home/bvissel/trajectory-analysis/trajectories/formatted_data/colgateaug0500summer2012080111.csv')


DistTravel_Aug2012 = firstTraj.distance_travelled()
print(latlon)
    

        
        
        


    



set0:42.82,-75.54
set0: base horizontal distance: 17.559219038408244
 new total distance: 17.559219038408244
set1:42.812,-75.755
set1: base horizontal distance: 17.340200318168698
 new total distance: 34.89941935657694
set2:42.78,-75.963
set2: base horizontal distance: 18.820300956647674
 new total distance: 53.71972031322461
set3:42.719,-76.178
set3: base horizontal distance: 22.048088980348137
 new total distance: 75.76780929357275
set4:42.63,-76.419
set4: base horizontal distance: 25.301273116254166
 new total distance: 101.06908240982692
set5:42.513,-76.684
set5: base horizontal distance: 28.86275754502854
 new total distance: 129.93183995485546
set6:42.364,-76.972
set6: base horizontal distance: 33.42680581457695
 new total distance: 163.35864576943243
set7:42.176,-77.289
set7: base horizontal distance: 36.78523626894314
 new total distance: 200.14388203837558
set8:41.957,-77.623
set8: base horizontal distance: 34.36035730280099
 new total distance: 234.50423934117657
set9:41.739,

In [7]:
import pandas as pd
import numpy as np
from math import *

class TrajectoryCalculations:
    def __init__(self, timestamp, datafile):
        self.timestamp = timestamp
        self.datafile = pd.read_csv(datafile)
    
    def distance_travelled(self):        
        lat1 = self.datafile['lat'].iloc[0]
        lat2 = self.datafile['lat'].iloc[-1]
        print(lat1, lat2)
        #acces the 11th column for lon
        lon1 = self.datafile['lon'].iloc[0]
        lon2 = self.datafile['lon'].iloc[-1]
        print(lon1, lon2)

        #formula to calculate the distance
        #convert decimal degrees to radians
        lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

        #haversine formula



firstTraj = TrajectoryCalculations('Aug20120801', '/home/bvissel/trajectory-analysis/trajectories/formatted_data/colgateaug0500summer2012080111.csv')


latlon = firstTraj.distance_travelled()
print(latlon)
    

42.82 41.318
-75.54 -78.309
None
